# CPU provider readback and cross-version check
Attach the private matched payload and local pilot readback Datasets. No training jobs or Final evaluation.

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, shutil, subprocess, sys, time
from datetime import datetime, timezone
base=Path('/kaggle/working/hung-provider-readback-v1')
assert not base.exists(), 'Preserve existing output; do not run this version twice'
payloads=list(Path('/kaggle/input').rglob('provider-payload.zip.bin'))
assert len(payloads)==1, 'Attach exact private payload dataset'
payload=payloads[0]
helper=payload.parent/'hung_provider.py'
assert hashlib.sha256(helper.read_bytes()).hexdigest()=='cadcd80d671d3b49a317312f125735362ffba3068247f1714a0ae56c6be83424'
assert hashlib.sha256(payload.read_bytes()).hexdigest()=='23a6d420523d990d07ff07d814ed642a851925283a102a3ae806f3b666e1d6f4'
spec=importlib.util.spec_from_file_location('hung_provider',helper)
provider=importlib.util.module_from_spec(spec)
spec.loader.exec_module(provider)
repo,plan_path=provider.materialize(payload,'23a6d420523d990d07ff07d814ed642a851925283a102a3ae806f3b666e1d6f4',base)
os.chdir(repo)
sys.path.insert(0,str(repo))
import torch
print(json.dumps({'torch':torch.__version__,'device':'cpu','scope':'Storage readback and cross-version reference only; no training jobs'}))


In [ ]:
"""Bounded CPU/CUDA portability check on two permitted initial Development examples."""
import json
from copy import deepcopy
import time
from pathlib import Path

import torch

from experiments.hung_sprint2 import dispatch_matched as dispatch
from experiments.hung_sprint2.cli import _examples, _tensors
from experiments.hung_sprint2.dispatch_state import atomic_create
from experiments.hung_sprint2.model import ForecastGRU, ModelConfig
from experiments.hung_sprint2.retention import retention_loss

torch.set_num_threads(1)
ATOL, RTOL = 1e-5, 1e-4
root = Path.cwd()
plan_path = root / '.cache/provider/plan.json'
plan = dispatch.read_verified_json(plan_path,dispatch.sha256_file(plan_path))
job = next(row for row in plan['jobs'] if row['id']=='online_best_b14s7e1153')
cell = next(row for row in plan['cells'] if row['id']==job['cell'])
config_pin = next(row for row in plan['configs'] if row['id']==job['config'])
config = dispatch.read_verified_json(Path(config_pin['path']),config_pin['sha256'])
input_run=Path(plan['input_base'])/'urbanev-SZH-14d'
manifest=dispatch._load_run(input_run,cell['input_manifest_sha256'])
rows,count=_examples(input_run,manifest,'development',cell['entity_id'],limit=2)
assert rows
x,c,y=_tensors(rows)
states={}
for name in ('source','target'):
    dispatch.verify_file(Path(job[name]['path']),job[name]['sha256'])
    states[name]=torch.load(job[name]['path'],map_location='cpu',weights_only=True)

def evaluate(device):
    target=ForecastGRU(ModelConfig(**states['target']['architecture'])).to(device).train()
    source=ForecastGRU(ModelConfig(**states['source']['architecture'])).to(device).eval()
    target.load_state_dict(states['target']['model_state'])
    source.load_state_dict(states['source']['model_state'])
    source.requires_grad_(False)
    optimizer=torch.optim.Adam(target.parameters(),lr=config['target']['learning_rate'])
    optimizer.load_state_dict(deepcopy(states['target']['optimizer_state']))
    prediction,latent=target.forecast_and_latent(x.to(device),c.to(device))
    with torch.no_grad():
        anchor=source.encode(x.to(device))
    forecast=(prediction-y.to(device)).abs().mean()
    retain=retention_loss(latent,anchor,states['source']['latent_scale'].to(device),epsilon=config['target']['epsilon'])
    loss=forecast+config['target']['retention_weight']*retain
    loss.backward()
    result={'forecast':prediction.detach().cpu(),'latent':latent.detach().cpu(),
            'forecast_loss':forecast.detach().cpu(),'retention_loss':retain.detach().cpu(),
            **{'gradient/'+name:parameter.grad.detach().cpu().clone() for name,parameter in target.named_parameters()}}
    optimizer.step()
    result.update({'updated/'+name:parameter.detach().cpu().clone() for name,parameter in target.named_parameters()})
    return result



In [ ]:
import zipfile
started=time.perf_counter()
archives=list(Path('/kaggle/input').rglob('pilot-artifacts.zip.bin'))
assert len(archives)==1
archive=archives[0]
assert archive.stat().st_size==101854497
assert provider.sha256(archive)=='c567611587a247a29ba72ad01cfe700c8296c2609126d117a634609c5c26e833'
with zipfile.ZipFile(archive) as stream:
 members=provider.safe_members(stream)
 assert len(members)==485
 job_receipts=[n for n in members if n.startswith('receipts/') and n.endswith('/job.json')]
 assert len(job_receipts)==6
storage={'archive_sha256':provider.sha256(archive),'bytes':archive.stat().st_size,
 'members':len(members),'job_receipts':len(job_receipts),'crc':'passed',
 'provider_mounted_remote_readback_seconds':time.perf_counter()-started,
 'scope':'Readback of persisted private Dataset on a different machine; not a local download or teammate review'}
provider.save(base/'pilot-storage-readback.json',storage)
reference=archive.parent/'cpu-reference.pt.bin'
assert provider.sha256(reference)=='10f4cee9ef8afcda2a27024136f08a9480ffc7116cec02f7215564cafca0ebf5'
expected=torch.load(reference,map_location='cpu',weights_only=True)
actual=evaluate('cpu')
assert expected.keys()==actual.keys()
checks={key:{'max_absolute_difference':float((actual[key]-expected[key]).abs().max()),
 'within_tolerance':bool(torch.allclose(actual[key],expected[key],atol=1e-5,rtol=1e-4))} for key in expected}
passed=all(row['within_tolerance'] for row in checks.values())
report={'passed':passed,'checks':checks,'reference_torch':'2.8.0+cu126','provider_torch':torch.__version__,
 'reference_sha256':provider.sha256(reference),'atol':1e-5,'rtol':1e-4,
 'code_sha':dispatch._code_identity()['commit'],'scope':'Bounded initial Development CPU comparison across Torch versions; no full-trajectory or performance equivalence claim'}
provider.save(base/'cross-runtime-parity.json',report)
print(json.dumps({'storage_readback':storage,'cross_runtime':report}))
assert passed,'Cross-version numerical reference mismatch; preserve outputs and investigate'
